Testing minkipy library at https://github.com/BAUDOTlab/minkiPy/tree/main

In [2]:
# imports

import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import minkiPy as mk



In [3]:
# data format
path = '/home/asalmona/Documents/Ricci/code/Celullar-Tissue-Spatial-Metrics-/data/pd_true_copy_with_genes.csv'
df = pd.read_csv(path)

In [4]:
# turn cell into a gen format 
def turn_cell_into_gen_format(
    df,
    noise_scale=0.0001,
    *,
    gene_cols=None,
    coord_cols=("coord_X", "coord_Y"),
    meta_cols=("cell_id", "cell_class", "coord_X", "coord_Y"),
    random_state=None,
    count_mode="auto",
    log2_offset=1.0,
    max_total_molecules=5_000_000,
):
    """
    Convert a cell-by-gene dataframe into one row per RNA molecule.

    The returned dataframe has columns: gene, global_x, global_y.

    count_mode:
        - "auto": invert log2(count + log2_offset) only when the resulting
          number of rows is manageable; otherwise round the current values.
        - "log2": strictly use 2**value - log2_offset.
        - "rounded": round the current gene values to integer counts.
    """
    x_col, y_col = coord_cols
    required_cols = {x_col, y_col}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise KeyError(f"Missing coordinate columns: {sorted(missing_cols)}")

    if noise_scale < 0:
        raise ValueError("noise_scale must be >= 0")

    if count_mode not in {"auto", "log2", "rounded"}:
        raise ValueError('count_mode must be one of "auto", "log2", or "rounded"')

    if gene_cols is None:
        gene_cols = [col for col in df.columns if col not in set(meta_cols)]
    else:
        gene_cols = list(gene_cols)

    if not gene_cols:
        return pd.DataFrame(columns=["gene", "global_x", "global_y"])

    gene_values = df[gene_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    gene_values = np.nan_to_num(gene_values, nan=0.0, posinf=0.0, neginf=0.0)

    def rounded_counts(values):
        return np.clip(np.rint(values), 0, None).astype(np.int64)

    def inverse_log2_counts(values):
        counts_float = np.rint(np.power(2.0, values) - log2_offset)
        counts_float = np.clip(counts_float, 0, None)
        total = counts_float.sum(dtype=np.float64)
        if total > max_total_molecules:
            raise ValueError(
                f"Inverse log2 would create {total:.3g} rows, which is above "
                f"max_total_molecules={max_total_molecules}. The values in this "
                "CSV look too large for a direct log2(count + 1) inverse. Use "
                'count_mode="rounded" or increase max_total_molecules if this is expected.'
            )
        return counts_float.astype(np.int64)

    if count_mode == "rounded":
        counts = rounded_counts(gene_values)
    elif count_mode == "log2":
        counts = inverse_log2_counts(gene_values)
    else:
        try:
            counts = inverse_log2_counts(gene_values)
        except ValueError:
            counts = rounded_counts(gene_values)

    total_molecules = int(counts.sum(dtype=np.int64))
    if total_molecules > max_total_molecules:
        raise ValueError(
            f"The output would contain {total_molecules:,} rows, which is above "
            f"max_total_molecules={max_total_molecules}. Increase the limit if needed."
        )

    cell_idx, gene_idx = np.nonzero(counts > 0)
    molecule_counts = counts[cell_idx, gene_idx]
    if molecule_counts.size == 0:
        return pd.DataFrame(columns=["gene", "global_x", "global_y"])

    repeated_cell_idx = np.repeat(cell_idx, molecule_counts)
    repeated_gene_idx = np.repeat(gene_idx, molecule_counts)

    coords = df[[x_col, y_col]].to_numpy(dtype=float)
    rng = np.random.default_rng(random_state)
    noise = rng.normal(loc=0.0, scale=noise_scale, size=(len(repeated_cell_idx), 2))
    rna_coords = coords[repeated_cell_idx] + noise

    return pd.DataFrame(
        {
            "gene": np.asarray(gene_cols, dtype=object)[repeated_gene_idx],
            "global_x": rna_coords[:, 0],
            "global_y": rna_coords[:, 1],
        }
    )

In [5]:
df_counts = turn_cell_into_gen_format(df)
print(df_counts.head())

/tmp/ipykernel_417086/1827384422.py:40: RuntimeWarning: invalid value encountered in cast
  counts = np.clip(counts, 0, None).astype(np.int64)


ValueError: negative dimensions are not allowed